# Overview

This notebook is for the purpose of analyzing the results from run.py when we run and collect raw trajectories, without any memories. This data is then used to create memories. This notebook contains code to also get (environment cue, LLM reasoning, action) tuples for each collected trajectory, for later storing in QDrant DB.

In [21]:
import pandas as pd
import csv

There are multiple runs for a given LLM. Let's consolidate

In [148]:
paths = ["runs/20251114114452_gpt/results.csv", "runs/20251115094638_gpt/results.csv", "runs/20251114173357_qwen/results.csv", "runs/20251115104018_qwen/results.csv"]

In [ ]:
# The saved CSV need to be reformatted a bit to properly load into a DF
def clean_csv(p):
    cleaned_rows = []

    with open(p, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)  # task,success,steps
        cleaned_rows.append(header)

        for row in reader:
            # If the row doesn't have exactly 3 columns, reconstruct from last two
            if len(row) != 3:
                # success and steps are always last two values
                success = row[-2]
                steps = row[-1]
                # everything else belongs to the task string
                task = ",".join(row[:-2])
            else:
                task, success, steps = row

            # Escape internal quotes by doubling them per CSV rules
            task = task.replace('"', '""')

            cleaned_rows.append([task, success, steps])

    # Write out fixed CSV with proper quoting
    with open(p.replace(".csv", "_cleaned.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerows(cleaned_rows)

In [27]:
for p in paths:
    clean_csv(p)

In [149]:
# GPT
cleaned_paths = [p.replace(".csv", "_cleaned.csv") for p in paths[:2]]
df1 = pd.read_csv(cleaned_paths[0])
df2 = pd.read_csv(cleaned_paths[1])

In [67]:
# Combine the two DataFrames
combined = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates based on the "task" column
unique_df_gpt = combined.drop_duplicates(subset=["task"], keep="first")

# Inspect
unique_df.head(40), len(unique_df)

(                                                 task  success  steps
 0                                       Fork ChatGPT.        0      0
 1   Show me the route and driving time from Allent...        0     30
 2   DisLike all submissions created by PatientBuil...        0      4
 3   Among the top 10 post in ""books"" forum, show...        0      3
 4   Add the following users to my time tracking to...        0     30
 5   Gather the titles of Nintendo Switch Fortnite ...        0      0
 6   Start a private project AGISite with JEKYLL te...        0     30
 7   Invite Jakub K, Alex Dills, Alex Hutnik and Be...        0     30
 8   Add HONGJ Hawaiian Beach Outfits Set for Mens,...        1      9
 9   Change my reddit bio to ""Pro Python Developer...        0      1
 10  I previously ordered some a table lamp in May ...        0     30
 11             How many commits did Eric make on 3/2?        0      1
 12  Fill the ""contact us"" form in the site for a...        0     12
 13  S

In [43]:
# QWEN
cleaned_paths = [p.replace(".csv", "_cleaned.csv") for p in paths[2:]]
df3 = pd.read_csv(cleaned_paths[0])
df4 = pd.read_csv(cleaned_paths[1])
# Combine the two DataFrames
combined = pd.concat([df3, df4], ignore_index=True)

# Remove duplicates based on the "task" column
unique_df_qwen = combined.drop_duplicates(subset=["task"], keep="first")

# Inspect
unique_df.head(), len(unique_df)

(                                                task  success  steps
 0                                      Fork ChatGPT.        0      0
 1  Show me the route and driving time from Allent...        0     30
 2  DisLike all submissions created by PatientBuil...        0      4
 3  Among the top 10 post in ""books"" forum, show...        0      3
 4  Add the following users to my time tracking to...        0     30,
 95)

Note down what trajectories succeeded, and which ones succeeded for 1 LLM but not another

In [48]:
common_tasks = set(unique_df_gpt['task']).intersection(set(unique_df_qwen['task']))
len(common_tasks)

81

In [50]:
# Filter both DFs to only successful tasks
df_gpt_success = unique_df_gpt[unique_df_gpt['success'] == 1]
df_qwen_success = unique_df_qwen[unique_df_qwen['success'] == 1]

# Find common tasks that succeeded in BOTH dataframes
common_success_tasks = set(df1_success['task']).intersection(set(df2_success['task']))

In [55]:
# Print tasks succeeded by each LLM, and both
print("Tasks succeeded by GPT:", len(df_gpt_success))
print("Tasks succeeded by QWEN:", len(df_qwen_success))
print("Tasks succeeded by both:", len(common_success_tasks))
# Print actual tasks (bullet points)
print("\nTasks succeeded by GPT:")
for task in df_gpt_success['task']:
    print(f"- {task}")

print("\nTasks succeeded by QWEN:")
for task in df_qwen_success['task']:
    print(f"- {task}")

print("\nTasks succeeded by both:")
for task in common_success_tasks:
    print(f"- {task}")

Tasks succeeded by GPT: 13
Tasks succeeded by QWEN: 14
Tasks succeeded by both: 5

Tasks succeeded by GPT:
- Add HONGJ Hawaiian Beach Outfits Set for Mens, Summer Tropical Tree Printed Relaxed-fit Hawaii Shirts Shorts 2 Piece Suits to my wish list
- Set my gitlab status as Playing Badminton.
- Add Tide PODS Spring Meadow Scent HE Turbo Laundry Detergent Pacs, 81 Count to my wish list
- Delete all pending negative reviews for Circe fleece
- How many commits did Philip make in 2023/1?
- Among the top 10 post in ""books"" forum, is there any post talks about supporting local book stores? If so, tell me the organizations involved
- Cancel order 299
- Pull up the description page of Carnegie Mellon University on Map
- List the top 1 search terms in my store
- Get the order number of my most recent complete order 
- Add this product to my wishlist
- Among the top 10 post in ""books"" forum, show me the book names from posts that recommand a single book
- What is the price range of Canon phot

### Summarize observations

In [100]:
from dotenv import load_dotenv
# input path to .env file, which should contain TOGETHER_API_KEY
load_dotenv(override=True)

%load_ext autoreload
%autoreload 2

from tqdm import tqdm
import sys
sys.path.append('../')

from webarena.browser_env.helper_functions import get_action_description
from webarena.agent.prompts import PromptConstructor
from memory.manager import MemoryManager

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [80]:
MEMORY_COLLECTION_NAME = "webarena-cues"
memory_manager = MemoryManager(collection_name=MEMORY_COLLECTION_NAME)

Collection 'webarena-cues' already exists, using existing collection.


In [87]:
from webarena.agent.agent import PromptAgent
agent = PromptAgent(
    action_set_tag="id_accessibility_tree",
    model="together_ai/OpenAI/gpt-oss-120B",
    temperature=0.7,
    use_litellm=True,
    instruction_path="agent/prompts/raw/p_direct_id_actree_2s_no_na.py",
    verbose=False # set to True if we want to see the full prompt and agent response under the hood each time
)

In [ ]:
def summarize_trajectory(trajectory: list):
    observations_actions_reasonings = []

    for i in tqdm(
        range(0, len(trajectory) - 1, 2),
        desc="Steps",
        position=1,
        leave=False,
    ):
        observation_item = trajectory[i]
        next_action_item = trajectory[i + 1]

        # Summarize the current observation (cue)
        summarized_obs = memory_manager.summarize_webarena_observation(observation_item["observation"]["text"])

        # Describe the next action taken after this observation
        next_action_text = get_action_description(
            next_action_item, observation_item["info"]["observation_metadata"],
            action_set_tag="id_accessibility_tree",
            prompt_constructor=agent.prompt_constructor
        )

        observations_actions_reasonings.append((summarized_obs, next_action_text, next_action_item.get('llm_reasoning')))

    return observations_actions_reasonings


In [98]:
def get_summarized_trajectories(run_path: str):
    print(f"Summarizing trajectories from {run_path}...")
    # Create the output directory if it doesn't exist
    from pathlib import Path
    output_dir = Path(run_path) / "summarized_trajectories"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    pkls = os.listdir(f"{run_path}/trajectories/")
    for pkl in tqdm(pkls, desc="Trajectories", position=0):
        traj = pickle.load(open(f"{run_path}/trajectories/{pkl}", "rb"))
        oar = summarize_trajectory(traj)
        # Store the pickle
        pickle.dump(oar, open(f"{run_path}/summarized_trajectories/{pkl}", "wb"))

In [101]:
get_summarized_trajectories("runs/20251114114452_gpt")

Summarizing trajectories from runs/20251114114452_gpt...


Trajectories: 100%|██████████| 83/83 [21:57<00:00, 15.88s/it]


In [ ]:
get_summarized_trajectories("runs/20251115094638_gpt")

In [ ]:
get_summarized_trajectories("runs/20251114173357_qwen")

In [ ]:
get_summarized_trajectories("runs/20251115104018_qwen")

## Storing into QDrant

Lets now store the summarized trajectories into QDrant. This is 1 form of memory our recall agent will consider.

We will have different collection names for different kinds of memory.

In [102]:
import sys
sys.path.append('../')

from memory.manager import MemoryManager

In [137]:
MEMORY_COLLECTION_NAME = "webarena-cues"
memory_manager = MemoryManager(collection_name=MEMORY_COLLECTION_NAME)

Collection 'webarena-cues' already exists, using existing collection.


In [135]:
# Collect mapping of task_id (pickle file name) to task (column in CSV)
# So we dont rely on unreliable mapping between CSV row and position of pickle in directory
import json
task_id_to_task = {}

st_idx = 0
ed_idx = 200 # However many webarena tasks we will use
test_file_list = []
for i in range(st_idx, ed_idx):
    test_file_list.append(f"config_files/{i}.json")

for config_file in test_file_list:
    with open(config_file) as f:
        _c = json.load(f)
        intent = _c["intent"]
        task_id = _c["task_id"]

        task_id_to_task[task_id] = intent

In [144]:
def store_summarized_trajectories_from_run(run_path: str):
    # Load the results csv
    df_results = pd.read_csv(f"{run_path}/results_cleaned.csv")
    df_results['task'] = df_results['task'].str.replace('""', '"')
    pickle_files = os.listdir(f"{run_path}/summarized_trajectories/")

    assert len(pickle_files) == len(df_results), "Number of pickle files must match number of results"

    for pickle_file in pickle_files:
        observations_actions_reasonings = pickle.load(open(f"{run_path}/summarized_trajectories/{pickle_file}", "rb"))

        task_id = int(pickle_file.replace(".pkl", ""))
        task = task_id_to_task[task_id]
        # print(task_id, task)
        # print(df_results['task'].values)
        
        row = df_results[df_results['task'] == task].values
        #assert len(row) == 1, f"Expected exactly one row for task: {task} (ID = {task_id})"
        assert len(row) > 0, f"No row found for task: {task} (ID = {task_id})"
        row = row[0]

        goal, success = row[0], row[1]

        memory_manager.store_trajectory(
            observations_actions_reasonings=observations_actions_reasonings,
            goal=goal,
            success=success == 1,
        )

In [ ]:
# Load the summarized trajectories...
remaining_paths = ["runs/20251114114452_gpt"]
#remaining_paths = ["runs/20251115094638_gpt", "runs/20251114173357_qwen", "runs/20251115104018_qwen"]

for p in remaining_paths:
    store_summarized_trajectories_from_run(p)

In [157]:
memory_manager.reset_database()

🗑️  Deleted collection 'webarena-cues'
✅ Recreated empty collection 'webarena-cues'
✅ Created payload index on 'goal' field for collection 'webarena-cues'


## Getting several sample memories by looking up based on goal/task

To be used to test RL training

In [174]:
df1[df1['success'] == 1].sample(1).values[0][0]

'Add Tide PODS Spring Meadow Scent HE Turbo Laundry Detergent Pacs, 81 Count to my wish list'

In [154]:
df1[df1['success'] == 0].sample(1).values[0][0]

'How many commits did Nic make in April 2021?'

In [175]:
g1 = 'Add Tide PODS Spring Meadow Scent HE Turbo Laundry Detergent Pacs, 81 Count to my wish list'
g2 = 'How many commits did Nic make in April 2021?'

In [186]:
def print_cue_mems_for_goal(goal: str):
    mems = memory_manager.get_memories_by_goal(goal)

    formatted_mems = []
    for m in mems:
        # If there is an 'obs_summary' field, then we know it's a cue-action mapping
        if 'obs_summary' in m:
            success = "success" if m['success'] else "failure"
            pointer = "(DONT DO AGAIN)" if not m['success'] else ""
            formatted_mems.append(f"""Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to {success}:

WHAT I SAW:
{m['obs_summary']}

WHAT I DID{pointer}:
{m['action_taken']}
""")

    print(f"MEMORIES FOR GOAL: {goal} (SUCCESS = {success})\n")
    for mem in formatted_mems:
        print(mem)
        print("--------------------------------")

In [187]:
print_cue_mems_for_goal(g1)

MEMORIES FOR GOAL: Add Tide PODS Spring Meadow Scent HE Turbo Laundry Detergent Pacs, 81 Count to my wish list (SUCCESS = success)

Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to success:

WHAT I SAW:
**Website:** One Stop Market – Home page (http://18.117.150.206:7770/)  

**Overall layout**
- Header with account links (“My Account”, “My Wish List”, “Sign Out”), a clickable store logo, a “My Cart” icon, and a search bar (combo‑box + “Search” button, currently disabled).  
- Global navigation presented as a horizontal tab list → vertical menu containing 12 top‑level categories (Beauty & Personal Care, Sports & Outdoors, Clothing Shoes & Jewelry, Home & Kitchen, Office Products, Tools & Home Improvement, Health 

WHAT I DID:
Attempt to perfom "type" on element "[298]" but no matching element found. Please check the observation more carefully.

--------------------------------
Last time I was in a similar situation, I tried doing 

In [189]:
print_cue_mems_for_goal(g2)

MEMORIES FOR GOAL: How many commits did Nic make in April 2021? (SUCCESS = failure)

Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to failure:

WHAT I SAW:
**Page Overview – “The A11Y Project / a11yproject.com” (GitLab project page)**  

| Category | Details |
|----------|---------|
| **Project** | **a11yproject.com** – community‑driven effort to simplify digital accessibility

WHAT I DID(DONT DO AGAIN):
Attempt to perfom "click" on element "[671]" but no matching element found. Please check the observation more carefully.

--------------------------------
Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to failure:

WHAT I SAW:
**Page Overview**  
- **Title / Context:** *Commits · main

WHAT I DID(DONT DO AGAIN):
scroll [down]

--------------------------------
Last time I was in a similar situation, I tried doing the corresponding action, and it ultimately led to failure:

WHAT